# 第9回：前処理をPipelineにまとめる

**今日の問い：数値列とカテゴリ列を、安全に同じモデルへ入れるにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
categorical = ["solvent", "catalyst", "scaffold_group"]
X = df[numeric + categorical]
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## TRY：列ごとの前処理を組み立てる


In [ ]:
numeric_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="median")),
    ("標準化", StandardScaler()),
])
categorical_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("数値列", numeric_process, numeric),
    ("カテゴリ列", categorical_process, categorical),
])
model = Pipeline([
    ("前処理", preprocess),
    ("予測", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
print(classification_report(y_valid, model.predict(X_valid), target_names=["非活性", "活性"]))


## 未知カテゴリでも予測できるか


In [ ]:
unknown = X_valid.iloc[[0]].copy()
unknown["solvent"] = "New-Solvent"
print("未知カテゴリを含む予測:", model.predict(unknown)[0])


## CHANGE

数値の欠損補完を`median`から`mean`へ変えます。変更はPipelineの1行だけにし、同じ検証データで比べます。
